# Silver Layer — Cleaning, Casting & Deduplication
**What this does:** Reads the 3 Bronze tables, cleans them, fixes data types, extracts nested fields, removes duplicates, and writes clean Silver tables ready for Gold/dbt.

**Why Silver exists:** Bronze is raw — types are wrong (timestamps stored as plain text), nested JSON is still locked inside string columns, and duplicate rows can exist if the pipeline ran twice. Silver fixes all of that. Think of it as the X-ray machine that inspects every bag and fixes broken tags before sending them to the next belt.

In [0]:
# functions as F = Spark's (distributed processing engine) built-in column functions
# F.col()               = reference a column by name
# F.col().cast()        = convert a column to a different type (string → timestamp, etc.)
# F.get_json_object()   = reach inside a JSON string and pull out one specific field
# .dropna()             = remove rows where critical columns are null/missing
# .dropDuplicates()     = keep only one row when two rows are identical on specified columns
from pyspark.sql import functions as F

In [0]:
# ── SILVER PRICES ─────────────────────────────────────────────────────────────
# Read the raw Bronze table — still has wrong types and all 26 columns
df_prices_bronze = spark.table("bronze_prices")

df_silver_prices = df_prices_bronze.select(

    # Identity columns — who is this coin
    F.col("id"),
    F.col("symbol"),
    F.col("name"),

    # Price metrics — cast to double (decimal number) because they came in as strings
    # WHY double? Prices like 65777.53 need decimal precision. "integer" would lose cents.
    F.col("current_price").cast("double"),
    F.col("market_cap").cast("double"),
    F.col("market_cap_rank").cast("integer"),
    F.col("total_volume").cast("double"),
    F.col("high_24h").cast("double"),
    F.col("low_24h").cast("double"),
    F.col("price_change_24h").cast("double"),
    F.col("price_change_percentage_24h").cast("double"),
    F.col("circulating_supply").cast("double"),

    # Timestamp — cast from plain text "2026-06-17T03:36:30.999Z" to a real timestamp
    # WHY cast? As plain text, you can't sort by it, can't do date math, can't filter by range.
    # As a timestamp type, Spark knows it's a date and handles all of that automatically.
    F.col("last_updated").cast("timestamp")

) \
.dropna(subset=["id", "current_price", "last_updated"]) \
# dropna = remove any row where these critical columns are null
# WHY these 3? A price row with no coin ID, no price, or no timestamp is useless garbage.
.dropDuplicates(["id", "last_updated"])
# dropDuplicates = if the same coin appears twice at the same exact timestamp, keep only one
# WHY deduplicate on id + last_updated? That combination uniquely identifies one price snapshot.
# If the pipeline ran twice in the same minute, this prevents double-counting.

print(f"Silver prices rows: {df_silver_prices.count()}")
display(df_silver_prices)